# 03 Cleanup and evaluation

This notebook is for the last step only: compare raw OCR against cleaned OCR and check whether the cleanup stage is helping at all. I want that separation to stay clear, because the cleanup is not supposed to become the recognizer.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluate import evaluate_records, load_ground_truth_records
from src.postprocess_llm import HISTORICAL_OCR_PROMPT_TEMPLATE
from src.utils import read_jsonl

In [2]:
predictions = read_jsonl(ROOT / 'data' / 'predictions' / 'baseline_predictions.jsonl')
ground_truth = load_ground_truth_records(ROOT / 'data' / 'ground_truth' / 'ground_truth.jsonl')
evaluation = evaluate_records(predictions, ground_truth)
evaluation['summary']

{'prediction_pages': 5,
 'ground_truth_pages': 5,
 'evaluated_pages': 3,
 'missing_ground_truth_page_ids': ['buendia_instruccion_page_0004',
  'buendia_instruccion_page_0005'],
 'unused_ground_truth_page_ids': ['covarrubias_tesoro_lengua_page_0001',
  'covarrubias_tesoro_lengua_page_0002'],
 'avg_raw_cer': 1.1468290469041373,
 'avg_cleaned_cer': 1.142308447960629,
 'avg_raw_wer': 1.3386999619347577,
 'avg_cleaned_wer': 1.3050277093771727,
 'cleanup_helped_pages_cer': 2,
 'cleanup_hurt_pages_cer': 1}

The prompt is deliberately restrained. If it starts modernizing the text or paraphrasing, it stops being useful for this task.

In [3]:
sample_text = predictions[0]['raw_ocr']
prompt = HISTORICAL_OCR_PROMPT_TEMPLATE.format(ocr_text=sample_text)
print(prompt[:1200])


You are correcting OCR output from 17th-century printed Spanish text.

Preserve meaning and historical spelling where reasonable.
Fix obvious OCR mistakes only.
Do not modernize the spelling unless the OCR is clearly wrong.
Do not rewrite the passage.
Return cleaned transcription only.

OCR output:
4 HRX X 1.1%,
@0.74 3 INSTRUCTION
WWW.WWWW.W.W.M.
UNIMUM P 1 V W
LIAT TAKELL & CHRISTANA, Y POLITIC
LT 7/11 : 1 M VORTEANIA CONDIOS
THANK *** ********
CREASE 18: 1 1 1: 1 @ DISPUESTA, PRIMARIAN
. LAM V 7 - X 8,PA IOSSHOROS COLEIALES D
ITEL: 11-11-11-12 IMPERIAL COLEGIO DE NUEOTA SETOR:
(AM) ***
INCLI L. D. : 4.3 COMPAMA DE JESAS DE
ITEM 21-1, W/4 11W - 1W A4PT 2M
JALM WILL 7'1'R BANDLOM, AK
LIKE TAX 1.5 SUAUTHOR 6%
SR WWW WWW.W.W.W.L.M.COM
***
YAP:17.00 3 QUEN LARDIA ADKNANT
WW WWWW.W.A.COM.COM.
TIME JAM Y/CM, JL 1/4 X AMADE NINO LEVEL.
CONDENCENCA.
IT RM
IT TOTAL
1 MARNOND ALA
INVOICE # 1 @ GERONA: POR LAYNABKO. L
1,8 PERFORGAN,LABORO,MO 17/20

